# 論文数値実験の再現

酒井・高山「需要主導型の輸送ネットワーク形成：Hub and Spoke 構造の創発」の数値実験 (case (a)〜(f), N=17) を再現する notebook．

v2 (`hubspoke.py`) は Numba JIT 化と分枝限定法のメモリ効率改善 (skip-on-push, periodic heap rebuild, diff-based heap encoding, exact-leaf skip) を加えた改良版．v1 (`hubspoke_v1.py`，論文通りの実装) と最終 `opt_Z` および `SP_tree` が完全一致することを確認済み．

## 6 ケースのパラメータ

全ケース共通: `t = 1.0`, `nu = 10.0`, `d = 1.0`, `N = 17`．以下が `(phi, rho)` の組合せ：

| ケース | パターン | phi | rho | 論文 opt_Z |
|:-:|:--|---:|---:|---:|
| (a) | 直鎖              | 50  | 20  | 1360.000 |
| (b) | 一階層ハブ        | 50  | 10  | 1143.453 |
| (c) | 二階層ハブ        | 10  | 10  | 897.577 |
| (d) | 三階層ハブ        | 10  | 5   | 626.487 |
| (e) | 一階層ハブ その 2 | 50  | 2.5 | 813.215 |
| (f) | 二階層ハブ その 2 | 25  | 2.5 | 641.631 |


In [ ]:
import io
import time
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import hubspoke as hs

import importlib
_ = importlib.reload(hs)

## ケース定義と解法関数

In [ ]:
CASES = {
    "(a) 直鎖":              dict(phi=50.0, rho=20.0, t=1.0, nu=10.0, d=1.0, N=17),
    "(b) 一階層ハブ":        dict(phi=50.0, rho=10.0, t=1.0, nu=10.0, d=1.0, N=17),
    "(c) 二階層ハブ":        dict(phi=10.0, rho=10.0, t=1.0, nu=10.0, d=1.0, N=17),
    "(d) 三階層ハブ":        dict(phi=10.0, rho=5.0,  t=1.0, nu=10.0, d=1.0, N=17),
    "(e) 一階層ハブ その2":  dict(phi=50.0, rho=2.5,  t=1.0, nu=10.0, d=1.0, N=17),
    "(f) 二階層ハブ その2":  dict(phi=25.0, rho=2.5,  t=1.0, nu=10.0, d=1.0, N=17),
}

# 論文掲載値 (solution.txt より) — 検証用
PAPER_OPT_Z = {
    "(a) 直鎖":              1360.0,
    "(b) 一階層ハブ":        1143.4527709275762,
    "(c) 二階層ハブ":        897.5771638137011,
    "(d) 三階層ハブ":        626.4871503404711,
    "(e) 一階層ハブ その2":  813.214765098425,
    "(f) 二階層ハブ その2":  641.6308408029813,
}

PAPER_SP_TREE = {
    "(a) 直鎖":              [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    "(b) 一階層ハブ":        [-1, 0, 1, 2, 3, 6, 7, 8, 0, 8, 9, 10, 11, 12, 13, 14, 15],
    "(c) 二階層ハブ":        [-1, 0, 1, 4, 5, 0, 5, 6, 9, 10, 5, 10, 11, 12, 13, 14, 15],
    "(d) 三階層ハブ":        [-1, 0, 1, 4, 5, 0, 5, 6, 9, 5, 9, 10, 13, 9, 13, 14, 15],
    "(e) 一階層ハブ その2":  [-1, 0, 1, 2, 3, 4, 7, 8, 9, 10, 0, 10, 11, 12, 13, 14, 15],
    "(f) 二階層ハブ その2":  [-1, 0, 1, 2, 5, 6, 7, 0, 7, 8, 9, 12, 13, 7, 13, 14, 15],
}


def solve_case(params, verbose=False):
    """一ケースを解いて (Parameter, Network, opt_Z, SP_tree, flow, 経過秒) を返す."""
    prm = hs.Parameter(**params)
    net = hs.Network(prm)
    init_lower = np.zeros(prm.L)
    init_upper = np.ones(prm.L) * (prm.nu * (prm.N - 2))
    model = hs.BB_model(prm, net)

    sink = io.StringIO()
    t0 = time.perf_counter()
    if verbose:
        opt_Z, opt_SP_tree, opt_flow = model.solve(init_lower, init_upper)
    else:
        with redirect_stdout(sink):
            opt_Z, opt_SP_tree, opt_flow = model.solve(init_lower, init_upper)
    elapsed = time.perf_counter() - t0
    return prm, net, float(opt_Z), np.asarray(opt_SP_tree), np.asarray(opt_flow), elapsed

## Numba JIT ウォームアップ

初回呼び出し時に Numba の JIT コンパイルが走る (数秒)．本番ケースの計測に影響しないよう，小規模ケースで一度だけウォームアップする．

In [ ]:
_warm = solve_case(dict(phi=50.0, rho=20.0, t=1.0, nu=10.0, d=1.0, N=4))
print(f"warm-up done in {_warm[-1]:.3f}s")

## 全 6 ケースを解く

case (d) (三階層ハブ) が最も重く 1〜3 分程度かかる．他のケースは数秒以内．

In [ ]:
results = {}
for label, params in CASES.items():
    print(f"solving {label} ...", flush=True)
    prm, net, optZ, SP_tree, flow, elapsed = solve_case(params)
    results[label] = dict(prm=prm, net=net, opt_Z=optZ,
                          SP_tree=SP_tree, flow=flow, time=elapsed)
    print(f"  opt_Z = {optZ:.9f}   time = {elapsed:.2f}s")

## 論文掲載値との照合

`opt_Z` と `SP_tree` が論文値と完全一致することを確認する．

In [ ]:
rows = []
for label, r in results.items():
    paper_z = PAPER_OPT_Z[label]
    paper_t = PAPER_SP_TREE[label]
    z_match = abs(r["opt_Z"] - paper_z) < 1e-9
    t_match = r["SP_tree"].tolist() == paper_t
    rows.append({
        "case":            label,
        "phi":             CASES[label]["phi"],
        "rho":             CASES[label]["rho"],
        "opt_Z":           r["opt_Z"],
        "paper opt_Z":     paper_z,
        "opt_Z match":     z_match,
        "SP_tree match":   t_match,
        "time (s)":        round(r["time"], 2),
    })
df = pd.DataFrame(rows)
df

## ネットワーク構造の可視化

各ケースの最適輸送ネットワーク．黒の弧が IT (近接) リンク，赤の弧が MT (集約) リンクで，赤いハブが現れる様子が確認できる．

In [ ]:
for label, r in results.items():
    print(f"\n=== {label}: opt_Z = {r['opt_Z']:.6f} ===")
    print(f"SP_tree = {r['SP_tree'].tolist()}")
    hs.link_plot(r["prm"], r["net"], r["SP_tree"])
    plt.title(label)
    plt.show()